In [ ]:
# ============================================================
# ONE-CELL RUNNER — Gemini 2.5 Pro (Vertex) TOP-K(=20)
# DRIVE-SAFE VERSION (fsync enabled + periodic close/reopen)
# ============================================================

import os, json, time
from tqdm import tqdm
from google.colab import drive, auth
from google import genai
from google.genai import types

# -----------------------------
# 0) Mount Drive + Auth (Vertex)
# -----------------------------
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted at /content/drive")

auth.authenticate_user()

# -----------------------------
# 1) Vertex client
# -----------------------------
PROJECT_ID = "project-b83c05da-b5d8-49a4-a2f"
LOCATION   = "us-central1"

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

MODEL_ID = "publishers/google/models/gemini-2.5-pro"
TOPK = 20
print(f"✅ Vertex client ready. MODEL_ID={MODEL_ID} TOPK={TOPK}")

# -----------------------------
# 2) Paths
# -----------------------------
TARGET_FILE = "clinical_pharm_prompts_10000.jsonl"
BASE_DIR = "/content/drive/MyDrive/DL_Local"
selected_path = os.path.join(BASE_DIR, TARGET_FILE)

SEED = 42
SPLIT_PATH = os.path.join(BASE_DIR, f"clinical_pharm_splits_random_8k_1k_1k_seed{SEED}.json")
OUTPUT_PATH = os.path.join(BASE_DIR, "clinical_pharm_teacher_topk_gemini25pro.jsonl")

# -----------------------------
# 3) Split behavior
# -----------------------------
GENERATE_FOR_VAL = True
N_TARGET = None

MAX_OUTPUT_TOKENS = 8000
MAX_TOKENS_TO_SAVE = None
SLEEP_SEC = 0.0

# ✅ NEW: close/reopen cadence (forces Drive to sync on close)
CLOSE_REOPEN_EVERY = 10  # set to 0/None to disable

# -----------------------------
# 4) Load dataset + splits
# -----------------------------
with open(selected_path, "r", encoding="utf-8") as f:
    first_ex = json.loads(next(f))
print("✅ Dataset OK. First id:", first_ex.get("id"))

with open(SPLIT_PATH, "r", encoding="utf-8") as f:
    splits = json.load(f)

train_ids = set(splits["train_ids"])
val_ids   = set(splits["val_ids"])
test_ids  = set(splits["test_ids"])

allowed_ids = train_ids | val_ids if GENERATE_FOR_VAL else train_ids

print("✅ Splits loaded.",
      "allowed:", len(allowed_ids),
      "test skipped:", len(test_ids))

# -----------------------------
# 5) Helpers
# -----------------------------
def extract_text(resp):
    if getattr(resp, "text", None):
        return resp.text
    try:
        for c in resp.candidates:
            if c.content and getattr(c.content, "parts", None):
                return "".join(p.text for p in c.content.parts if getattr(p, "text", None))
    except Exception:
        pass
    return None

def safe_usage(resp):
    um = getattr(resp, "usage_metadata", None)
    return {
        "prompt_tokens": getattr(um, "prompt_token_count", None),
        "thoughts_tokens": getattr(um, "thoughts_token_count", None),
        "total_tokens": getattr(um, "total_token_count", None),
    }

def finish_reason(resp):
    try:
        return str(resp.candidates[0].finish_reason)
    except Exception:
        return None

def already_done_ids(path):
    done = set()
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    if "id" in obj:
                        done.add(obj["id"])
                except Exception:
                    pass
    return done

def parse_logprobs(resp, max_steps=None):
    steps = []
    cand = resp.candidates[0]
    lpr = getattr(cand, "logprobs_result", None) or getattr(cand, "logprobsResult", None)
    if not lpr:
        return steps

    chosen = lpr.chosen_candidates
    top = lpr.top_candidates

    for i, ch in enumerate(chosen):
        if max_steps and i >= max_steps:
            break

        entry = {
            "chosen": {
                "token": ch.token,
                "token_id": getattr(ch, "token_id", None),
                "logprob": getattr(ch, "log_probability", None),
            },
            "topk": []
        }

        if top and i < len(top) and top[i]:
            for tc in top[i].candidates:
                entry["topk"].append({
                    "token": tc.token,
                    "token_id": getattr(tc, "token_id", None),
                    "logprob": getattr(tc, "log_probability", None),
                })

        steps.append(entry)

    return steps

# -----------------------------
# 6) SAFE generation loop
# -----------------------------
done_ids = already_done_ids(OUTPUT_PATH)
print(f"Already done: {len(done_ids)}")
print(f"📄 Writing to: {OUTPUT_PATH}")

written = 0
seen = 0

# NOTE: we keep fin in a context manager, but manage fout manually so we can close/reopen it.
with open(selected_path, "r", encoding="utf-8") as fin:
    fout = open(OUTPUT_PATH, "a", encoding="utf-8")  # ✅ manual open

    try:
        for line in tqdm(fin):
            ex = json.loads(line)
            ex_id = ex.get("id")
            if not ex_id:
                continue

            seen += 1

            if ex_id in test_ids or ex_id not in allowed_ids or ex_id in done_ids:
                continue

            prompt = ex["prompt"] + (
                "\n\nIMPORTANT:\n"
                "- Follow the exact required output format.\n"
                "- Keep it concise.\n"
                "- Do not add extra sections.\n"
            )

            try:
                resp = client.models.generate_content(
                    model=MODEL_ID,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0.0,
                        max_output_tokens=MAX_OUTPUT_TOKENS,
                        response_logprobs=True,
                        logprobs=TOPK,
                    ),
                )

                record = {
                    "id": ex_id,
                    "split": "train" if ex_id in train_ids else "val",
                    "status": "ok",
                    "model": MODEL_ID,
                    "topk": TOPK,
                    "finish_reason": finish_reason(resp),
                    "usage": safe_usage(resp),
                    "teacher_text": extract_text(resp),
                    "logprobs_steps": parse_logprobs(resp, MAX_TOKENS_TO_SAVE),
                }

            except Exception as e:
                record = {
                    "id": ex_id,
                    "split": "train" if ex_id in train_ids else "val",
                    "status": "error",
                    "error": repr(e),
                }

            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            fout.flush()
            os.fsync(fout.fileno())   # 🔒 HARD SYNC

            written += 1
            done_ids.add(ex_id)

            if written % 10 == 0:
                size_mb = os.path.getsize(OUTPUT_PATH) / (1024**2)
                print(f"✅ WROTE {written} | seen={seen} | file={size_mb:.2f} MB")

            # ✅ NEW: force Drive to commit by closing the file periodically
            if CLOSE_REOPEN_EVERY and (written % CLOSE_REOPEN_EVERY == 0):
                fout.close()
                fout = open(OUTPUT_PATH, "a", encoding="utf-8")

            if SLEEP_SEC:
                time.sleep(SLEEP_SEC)

            if N_TARGET and written >= N_TARGET:
                break

    finally:
        try:
            fout.close()
        except Exception:
            pass

print(f"✅ DONE. Written this run: {written}")
print(f"📄 Final size: {os.path.getsize(OUTPUT_PATH)/(1024**2):.2f} MB")


In [2]:
!cp /content/drive/MyDrive/DL_Local/clinical_pharm_teacher_topk_gemini25pro.jsonl \
    /content/drive/MyDrive/DL_Local/clinical_pharm_teacher_topk_gemini25pro_SAVED_2026-01-14.jsonl
!sync


In [5]:
# ============================================================
# COLAB CELL — Random train/val/test split (8k/1k/1k) + save to Drive
# - Randomly selects 1000 TEST and 1000 VAL (disjoint)
# - Remaining IDs are TRAIN
# - Writes JSON split + txt lists to /content/drive/MyDrive
# ============================================================

import os, json, random
from google.colab import drive

# 0) Mount Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted at /content/drive")

# 1) Paths
BASE_DIR = "/content/drive/MyDrive/DL_Local"
TARGET_FILE = "clinical_pharm_prompts_10000.jsonl"
DATASET_PATH = os.path.join(BASE_DIR, TARGET_FILE)

SEED = 42
N_TEST = 1000
N_VAL  = 1000

SPLIT_OUT_JSON = os.path.join(BASE_DIR, f"clinical_pharm_splits_random_8k_1k_1k_seed{SEED}.json")
TRAIN_TXT = os.path.join(BASE_DIR, f"clinical_pharm_train_ids_seed{SEED}.txt")
VAL_TXT   = os.path.join(BASE_DIR, f"clinical_pharm_val_ids_seed{SEED}.txt")
TEST_TXT  = os.path.join(BASE_DIR, f"clinical_pharm_test_ids_seed{SEED}.txt")

assert os.path.exists(DATASET_PATH), f"❌ Dataset not found: {DATASET_PATH}"

# 2) Load IDs (keep file order for reproducibility/debugging)
ids = []
seen = set()
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        ex = json.loads(line)
        ex_id = ex.get("id")
        if ex_id is None:
            continue
        # ensure uniqueness (if duplicates exist, first occurrence wins)
        if ex_id in seen:
            continue
        seen.add(ex_id)
        ids.append(ex_id)

N = len(ids)
print(f"✅ Loaded {N} unique IDs from dataset.")

if N < (N_TEST + N_VAL + 1):
    raise ValueError(f"Dataset too small: N={N} but need at least {N_TEST+N_VAL+1}")

# 3) Random split (disjoint)
rng = random.Random(SEED)
all_idx = list(range(N))
rng.shuffle(all_idx)

test_idx = set(all_idx[:N_TEST])
val_idx  = set(all_idx[N_TEST:N_TEST+N_VAL])

test_ids = [ids[i] for i in all_idx[:N_TEST]]
val_ids  = [ids[i] for i in all_idx[N_TEST:N_TEST+N_VAL]]

# Train = all remaining, keep original file order (useful)
train_ids = [ids[i] for i in range(N) if (i not in test_idx and i not in val_idx)]

# 4) Save
splits = {
    "dataset_file": TARGET_FILE,
    "strategy": "random_disjoint",
    "seed": SEED,
    "counts": {"train": len(train_ids), "val": len(val_ids), "test": len(test_ids)},
    "train_ids": train_ids,
    "val_ids": val_ids,
    "test_ids": test_ids,
}

with open(SPLIT_OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(splits, f, ensure_ascii=False, indent=2)

with open(TRAIN_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(map(str, train_ids)) + "\n")

with open(VAL_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(map(str, val_ids)) + "\n")

with open(TEST_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(map(str, test_ids)) + "\n")

print("✅ Wrote split JSON:", SPLIT_OUT_JSON)
print("✅ Wrote train IDs:", TRAIN_TXT)
print("✅ Wrote val IDs  :", VAL_TXT)
print("✅ Wrote test IDs :", TEST_TXT)
print("Counts:", splits["counts"])

# 5) Sanity checks
assert len(set(train_ids) & set(val_ids)) == 0
assert len(set(train_ids) & set(test_ids)) == 0
assert len(set(val_ids) & set(test_ids)) == 0
assert len(train_ids) + len(val_ids) + len(test_ids) == N

print("✅ Sanity checks passed (disjoint + total match).")


Drive already mounted at /content/drive
✅ Loaded 10000 unique IDs from dataset.
✅ Wrote split JSON: /content/drive/MyDrive/DL_Local/clinical_pharm_splits_random_8k_1k_1k_seed42.json
✅ Wrote train IDs: /content/drive/MyDrive/DL_Local/clinical_pharm_train_ids_seed42.txt
✅ Wrote val IDs  : /content/drive/MyDrive/DL_Local/clinical_pharm_val_ids_seed42.txt
✅ Wrote test IDs : /content/drive/MyDrive/DL_Local/clinical_pharm_test_ids_seed42.txt
Counts: {'train': 8000, 'val': 1000, 'test': 1000}
✅ Sanity checks passed (disjoint + total match).


In [1]:
from google.colab import auth
auth.authenticate_user()


In [3]:
!gcloud config set project project-b83c05da-b5d8-49a4-a2f


Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey



In [7]:
from google import genai
from google.genai import types

PROJECT_ID = "project-b83c05da-b5d8-49a4-a2f"
LOCATION = "us-central1"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

MODEL_ID = "publishers/google/models/gemini-3-pro-preview"

print("Vertex Gemini client ready")


Vertex Gemini client ready


In [6]:
models = client.models.list()
for m in models:
    print(getattr(m, "name", m))


publishers/google/models/imageclassification-efficientnet
publishers/google/models/occupancy-analytics
publishers/google/models/multimodalembedding
publishers/google/models/pt-test
publishers/google/models/imageclassification-vit
publishers/google/models/bert-base
publishers/google/models/vehicle-detector
publishers/google/models/language-v1-classify-text-v1
publishers/google/models/language-v1-analyze-sentiment
publishers/google/models/language-v1-analyze-entity-sentiment
publishers/google/models/language-v1-analyze-syntax
publishers/google/models/resnet50
publishers/google/models/imagesegmentation-deeplabv3
publishers/google/models/imageobjectdetection-yolo
publishers/google/models/owlvit-base-patch32
publishers/google/models/object-detector
publishers/google/models/ppe-detector
publishers/google/models/people-blur
publishers/google/models/product-recognizer
publishers/google/models/tag-recognizer
publishers/google/models/imageclassification-proprietary-vit
publishers/google/models/i

In [9]:
from google.genai import types
import re, json

MODEL_ID = "publishers/google/models/gemini-3-pro-preview"

resp = client.models.generate_content(
    model=MODEL_ID,
    contents="Say exactly: Decision: Safe",
    config=types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=128,
        response_logprobs=True,
        logprobs=10,   # try 10 first
    ),
)

print("TEXT:", getattr(resp, "text", None))
print("RAW:", resp)

# quick scan for any logprob fields in the python repr
s = str(resp)
print("\nLogprob-related strings found:", sorted(set(re.findall(r"logprob\\w*", s, flags=re.IGNORECASE))))


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher Model `projects/project-b83c05da-b5d8-49a4-a2f/locations/us-central1/publishers/google/models/gemini-3-pro-preview` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions', 'status': 'NOT_FOUND'}}

In [10]:
!gcloud config get-value project
!gcloud auth list


project-b83c05da-b5d8-49a4-a2f
  Credentialed Accounts
ACTIVE  ACCOUNT
*       gal2361@gmail.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`



In [11]:
from google.genai import types

MODEL_ID = "publishers/google/models/gemini-2.5-pro"  # stable + should be accessible

resp = client.models.generate_content(
    model=MODEL_ID,
    contents="Say exactly: Decision: Safe",
    config=types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=64,
    ),
)
print("OK. Got response.")
print(resp)


OK. Got response.
sdk_http_response=HttpResponse(
  headers=<dict len=9>
) candidates=[Candidate(
  avg_logprobs=-0.8600943883260092,
  content=Content(
    parts=[
      Part(
        text='Decision: Safe'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.MAX_TOKENS: 'MAX_TOKENS'>
)] create_time=datetime.datetime(2026, 1, 11, 10, 38, 41, 218112, tzinfo=TzInfo(0)) model_version='gemini-2.5-pro' prompt_feedback=None response_id='sX1jaYCoDfOIm9IPorKV-QE' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=3,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=3
    ),
  ],
  prompt_token_count=6,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=6
    ),
  ],
  thoughts_token_count=57,
  total_token_count=66,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
) automatic_function_calling_history=[] parsed=None


In [12]:
from google import genai
from google.genai import types

PROJECT_ID = "YOUR_PROJECT_ID"
MODEL_ID = "publishers/google/models/gemini-3-pro-preview"
REGIONS = ["us-central1", "us-east1", "us-west1", "europe-west4", "asia-southeast1"]

for region in REGIONS:
    try:
        c = genai.Client(vertexai=True, project=PROJECT_ID, location=region)
        r = c.models.generate_content(
            model=MODEL_ID,
            contents="ping",
            config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=16),
        )
        print(f"✅ WORKS in {region}")
        break
    except Exception as e:
        print(f"❌ {region}: {type(e).__name__}: {str(e)[:140]}")


❌ us-central1: ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Permission denied on resource project YOUR_PROJECT_ID.', 'status': 'PERMISSION_DE
❌ us-east1: ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Permission denied on resource project YOUR_PROJECT_ID.', 'status': 'PERMISSION_DE
❌ us-west1: ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Permission denied on resource project YOUR_PROJECT_ID.', 'status': 'PERMISSION_DE
❌ europe-west4: ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Permission denied on resource project YOUR_PROJECT_ID.', 'status': 'PERMISSION_DE
❌ asia-southeast1: ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Permission denied on resource project YOUR_PROJECT_ID.', 'status': 'PERMISSION_DE


In [13]:
# ============================================================
# COLAB CELL — Gemini 2.5 Pro (Vertex) + load JSONL + TOP-K(=20) tokens -> JSONL
# Writes incrementally (resume-safe), keeps original "id".
#
# NOTE: This can get VERY large on disk (top-20 per output token).
# You can cap tokens saved via MAX_TOKENS_TO_SAVE if needed.
# ============================================================
import os, json, time
from tqdm import tqdm
from google.colab import drive, auth
from google import genai
from google.genai import types

# ---------- 0) Mount Drive + Auth ----------
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted.")

auth.authenticate_user()

# ---------- 1) Configure Vertex client ----------
PROJECT_ID = "project-b83c05da-b5d8-49a4-a2f"   # <-- your GCP project
LOCATION   = "us-central1"

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

MODEL_ID = "publishers/google/models/gemini-2.5-pro"
TOPK = 20  # max supported on Vertex (1–20) per docs. :contentReference[oaicite:0]{index=0}

print(f"✅ Vertex client ready. MODEL_ID={MODEL_ID} TOPK={TOPK} project={PROJECT_ID} loc={LOCATION}")

# ---------- 2) Dataset paths ----------
TARGET_FILE = "clinical_pharm_prompts_10000.jsonl"
BASE_DIR = "/content/drive/MyDrive"  # adjust if needed
selected_path = os.path.join(BASE_DIR, TARGET_FILE)

if not os.path.exists(selected_path):
    raise FileNotFoundError(f"❌ Dataset not found: {selected_path}")

OUTPUT_PATH = "/content/drive/MyDrive/clinical_pharm_teacher_topk_gemini25pro.jsonl"

# How many examples to process now (set 10000 for full)
N_TARGET = 10000

# Model output budget (Gemini 2.5 is usually much less "thought-hungry" than Gemini 3)
MAX_OUTPUT_TOKENS = 10000

# Optional: cap how many generated tokens you SAVE logprobs for (storage control).
# Set to None to save all generated steps.
MAX_TOKENS_TO_SAVE = None   # e.g., 256 to reduce file size

# API politeness
SLEEP_SEC = 0.0  # set to 0.2–1.0 if you hit rate limits


# ---------- 3) Helpers ----------
def extract_text(resp):
    # Prefer resp.text if present
    if getattr(resp, "text", None):
        return resp.text
    # Fallback: candidates -> content.parts
    try:
        for c in resp.candidates:
            if c.content and getattr(c.content, "parts", None):
                chunks = []
                for p in c.content.parts:
                    t = getattr(p, "text", None)
                    if t:
                        chunks.append(t)
                if chunks:
                    return "".join(chunks)
    except Exception:
        pass
    return None

def safe_usage(resp):
    um = getattr(resp, "usage_metadata", None)
    return {
        "prompt_tokens": getattr(um, "prompt_token_count", None),
        "candidates_tokens": getattr(um, "candidates_token_count", None),
        "thoughts_tokens": getattr(um, "thoughts_token_count", None),
        "total_tokens": getattr(um, "total_token_count", None),
    }

def finish_reason(resp):
    try:
        return str(resp.candidates[0].finish_reason)
    except Exception:
        return None

def parse_logprobs(resp, max_steps=None):
    """
    Returns a list[dict] of decoding steps:
      step = {
        "chosen": {"token": str, "token_id": int|None, "logprob": float|None},
        "topk":   [{"token": str, "token_id": int|None, "logprob": float|None}, ...]
      }
    Compatible with Vertex "logprobsResult" / python-genai "logprobs_result".
    """
    steps = []

    cand0 = None
    try:
        cand0 = resp.candidates[0]
    except Exception:
        return steps

    lpr = getattr(cand0, "logprobs_result", None)
    if lpr is None:
        # Some SDK builds may use a different name; try a couple fallbacks
        lpr = getattr(cand0, "logprobsResult", None)
    if lpr is None:
        return steps

    chosen = getattr(lpr, "chosen_candidates", None) or getattr(lpr, "chosenCandidates", None)
    top    = getattr(lpr, "top_candidates", None)    or getattr(lpr, "topCandidates", None)

    if chosen is None:
        return steps

    # chosen: list[Candidate], top: list[TopCandidates] aligned by step
    for i in range(len(chosen)):
        if max_steps is not None and i >= max_steps:
            break

        ch = chosen[i]
        ch_dict = {
            "token": getattr(ch, "token", None),
            "token_id": getattr(ch, "token_id", None) if hasattr(ch, "token_id") else getattr(ch, "tokenId", None),
            "logprob": getattr(ch, "log_probability", None) if hasattr(ch, "log_probability") else getattr(ch, "logProbability", None),
        }

        topk_list = []
        if top is not None and i < len(top) and top[i] is not None:
            top_i = top[i]
            cand_list = getattr(top_i, "candidates", None)
            if cand_list is None:
                cand_list = getattr(top_i, "candidates_", None)
            if cand_list:
                for tc in cand_list:
                    topk_list.append({
                        "token": getattr(tc, "token", None),
                        "token_id": getattr(tc, "token_id", None) if hasattr(tc, "token_id") else getattr(tc, "tokenId", None),
                        "logprob": getattr(tc, "log_probability", None) if hasattr(tc, "log_probability") else getattr(tc, "logProbability", None),
                    })

        steps.append({"chosen": ch_dict, "topk": topk_list})

    return steps

def already_done_ids(path):
    done = set()
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        obj = json.loads(line)
                        if "id" in obj:
                            done.add(obj["id"])
                    except Exception:
                        pass
    return done


# ---------- 4) Resume-safe generation loop ----------
done_ids = already_done_ids(OUTPUT_PATH)
print(f"Already in output (will skip): {len(done_ids)}")

written = 0
with open(selected_path, "r", encoding="utf-8") as fin, open(OUTPUT_PATH, "a", encoding="utf-8") as fout:
    for line in tqdm(fin, total=N_TARGET):
        if written >= N_TARGET:
            break
        if not line.strip():
            continue

        ex = json.loads(line)
        ex_id = ex.get("id")
        if not ex_id:
            continue

        if ex_id in done_ids:
            continue

        # Keep your original prompt but add a strict formatting reminder
        prompt = ex["prompt"] + (
            "\n\nIMPORTANT:\n"
            "- Follow the exact required output format.\n"
            "- Keep it concise.\n"
            "- Do not add extra sections.\n"
        )

        try:
            resp = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                    response_logprobs=True,
                    logprobs=TOPK,
                ),
            )

            text = extract_text(resp)
            steps = parse_logprobs(resp, max_steps=MAX_TOKENS_TO_SAVE)

            record = {
                "id": ex_id,
                "status": "ok" if text is not None else "no_text",
                "model": MODEL_ID,
                "topk": TOPK,
                "finish_reason": finish_reason(resp),
                "usage": safe_usage(resp),
                "teacher_text": text,
                "logprobs_steps": steps,  # huge; each step includes chosen token + top-k alternatives
            }

        except Exception as e:
            record = {
                "id": ex_id,
                "status": "error",
                "model": MODEL_ID,
                "topk": TOPK,
                "error": repr(e),
            }

        fout.write(json.dumps(record, ensure_ascii=False) + "\n")
        fout.flush()

        written += 1
        done_ids.add(ex_id)

        if SLEEP_SEC > 0:
            time.sleep(SLEEP_SEC)

print(f"✅ Done. Newly written this run: {written}")
print(f"📄 Output JSONL: {OUTPUT_PATH}")


Mounted at /content/drive
✅ Vertex client ready. MODEL_ID=publishers/google/models/gemini-2.5-pro TOPK=20 project=project-b83c05da-b5d8-49a4-a2f loc=us-central1
Already in output (will skip): 0


  0%|          | 30/10000 [09:03<50:09:36, 18.11s/it]


KeyboardInterrupt: 

In [ ]:
import os, glob

print("OUTPUT_PATH =", OUTPUT_PATH)
print("Exists OUTPUT_PATH? ", os.path.exists(OUTPUT_PATH))
print("Parent exists? ", os.path.exists(os.path.dirname(OUTPUT_PATH)))

print("\nList /content/drive/MyDrive (first 50):")
print(os.listdir("/content/drive/MyDrive")[:50])

print("\nSearch anywhere under /content/drive for the output filename:")
matches = glob.glob("/content/drive/**/clinical_pharm_teacher_topk_gemini25pro.jsonl", recursive=True)
print("Matches:", matches)


OUTPUT_PATH = /content/drive/MyDrive/clinical_pharm_teacher_topk_gemini25pro.jsonl
Exists OUTPUT_PATH?  True
Parent exists?  True

List /content/drive/MyDrive (first 50):
['משבר כלכלי באירופה.docx', 'ללא שם (1).gmap', 'ללא שם.gmap', 'IMG_5845.JPG', 'מבחן 1.docx.gdoc', 'Master Page & Menu 58.pptx', 'פתרון מספר 1.gdoc', 'What are the qualities that make a good teacher.docx', 'Body.docx', 'צוות שיווק ובוגרים.gslides', 'IMG_0786.MP4', 'מכתבים', 'יובל צבי.gdoc', 'אביסרור.gdoc', 'טל וייס .gdoc', 'פריד.gdoc', 'שושני .gdoc', 'טהיר.gdoc', 'גולי.gdoc', 'שיר חנוכה.m4a', '2017-12', 'F7C0DD87-B7F5-462B-9914-39B79C56A8F8.JPG', '56.jpg', 'יודנראט.gdoc', '\u200fהקלטה חדשה \u200f3.m4a', '\u200fהקלטה חדשה \u200f4.m4a', '097A2DB9-531D-4436-BD09-A5144DA4E473.JPG', 'IMG_1536.MOV', 'קטע סיום.gdoc', 'רגע שיא.gdoc', '56995f14-539b-45a2-a1fa-0c3fbad8fc1f.jpg', 'IMG_2446.PNG', 'עדי גוט סוף.gdoc', 'המנטורים.gdoc', '4C4B04A1-985F-444D-B79F-2B2D027F35B2.JPG', 'FEC128C2-8865-43A4-986D-8413389DBA99.JPG', '290EDCFC-2